In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
import numpy as np

print("Tools loaded. Ready to build.")

Tools loaded. Ready to build.


In [2]:
# Load the data already split into Training and Testing sets
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)

print(f"Training reviews: {len(x_train)}")
print(f"Testing reviews: {len(x_test)}")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Training reviews: 25000
Testing reviews: 25000


In [3]:
# Make all sequences 200 words long
x_train = pad_sequences(x_train, maxlen=200)
x_test = pad_sequences(x_test, maxlen=200)

print("Reviews padded to 200 words.")

Reviews padded to 200 words.


In [5]:
# Updated Step 4: The Clean Version
model = Sequential([
    # We removed 'input_length' to stop the warning
    Embedding(input_dim=10000, output_dim=128),
    
    # LSTM Layer for sequence memory
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    
    # Final decision layer
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', 
              optimizer='adam', 
              metrics=['accuracy'])

# This prints the table that proves your model is built correctly
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
print("Starting training... this may take 1-2 minutes.")
model.fit(x_train, y_train, batch_size=32, epochs=3, validation_data=(x_test, y_test))

Starting training... this may take 1-2 minutes.
Epoch 1/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 140s 171ms/step - accuracy: 0.7880 - loss: 0.4562 - val_accuracy: 0.8395 - val_loss: 0.3752
Epoch 2/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 122s 156ms/step - accuracy: 0.8459 - loss: 0.3612 - val_accuracy: 0.8069 - val_loss: 0.4342
Epoch 3/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 125s 160ms/step - accuracy: 0.8879 - loss: 0.2765 - val_accuracy: 0.8501 - val_loss: 0.3548


In [7]:
loss, accuracy = model.evaluate(x_test, y_test)
print(f"Final Model Accuracy: {accuracy * 100:.2f}%")

782/782 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.8501 - loss: 0.3548
Final Model Accuracy: 85.01%


In [8]:
# Function to prepare your own text for the model
def predict_sentiment(my_review):
    # Get the word index from the IMDB dataset
    word_index = imdb.get_word_index()
    # Convert words to the numbers the model understands
    words = my_review.lower().split()
    review_seq = [word_index.get(w, 0) + 3 for w in words] # +3 is an IMDB specific offset
    # Pad to 200 words
    review_padded = pad_sequences([review_seq], maxlen=200)
    # Predict!
    prediction = model.predict(review_padded)
    sentiment = "Positive" if prediction > 0.5 else "Negative"
    print(f"Review: {my_review}")
    print(f"Model Prediction: {sentiment} ({prediction[0][0]*100:.2f}%)")

# Test it!
predict_sentiment("this movie was amazing and the acting was great")
predict_sentiment("it was a waste of time and very boring")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 2s 1us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 700ms/step
Review: this movie was amazing and the acting was great
Model Prediction: Positive (92.50%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Review: it was a waste of time and very boring
Model Prediction: Negative (6.84%)


In [9]:
model.save("my_sentiment_model.keras")
print("Model saved successfully as 'my_sentiment_model.keras'!")

Model saved successfully as 'my_sentiment_model.keras'!
